# Review counts per language, per user (users with at least one English review)

Goal: for each user who wrote at least one English review, count how many reviews they wrote in each language.

In [ ]:
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from pathlib import Path

ROOT = Path.cwd()
REVIEWS_DATA = ROOT / 'data' / 'corpus' / 'reviews_by_lang'
EN_USERS_LANG_COUNTS_FILE = ROOT / 'data' / 'features' / 'en_users_language_counts.parquet'

## 1. Available languages

Discovers the existing partitions in `REVIEWS_DATA` (it does not assume a fixed language list; it uses whatever is on disk).

In [ ]:
lang_dirs = sorted(p for p in REVIEWS_DATA.glob('review_lang=*') if p.is_dir())
langs = [d.name.split('=', 1)[1] for d in lang_dirs]
print(f'{len(langs)} languages found: {langs}')

## 2. Users with at least one English review

Reads only the `user_url` column from every `review_lang=en` file (avoids loading `review_text` and other columns).

In [ ]:
en_dir = REVIEWS_DATA / 'review_lang=en'
en_files = sorted(en_dir.glob('*.parquet'))
print(f'{len(en_files)} files in review_lang=en')

t0 = time.time()
en_table = pa.concat_tables([pq.read_table(f, columns=['user_url']) for f in en_files])
n_en_reviews = en_table.num_rows
en_users = en_table.column('user_url').combine_chunks().unique()
del en_table

print(f'English reviews: {n_en_reviews:,}')
print(f'Unique users with >= 1 English review: {len(en_users):,}')
print(f'Time: {time.time() - t0:.1f}s')

## 3. Per-language review counts, restricted to English users

For each language: read the `user_url` column, keep only the rows whose user is in `en_users`, and count reviews per user.

In [ ]:
long_tables = []

for lang_dir, lang in zip(lang_dirs, langs):
    files = sorted(lang_dir.glob('*.parquet'))
    tbl = pa.concat_tables([pq.read_table(f, columns=['user_url']) for f in files])

    mask = pc.is_in(tbl.column('user_url'), value_set=en_users)
    filtered = tbl.filter(mask)
    del tbl, mask

    vc = filtered.column('user_url').combine_chunks().value_counts()
    del filtered

    n_users_lang = len(vc)
    lang_tbl = pa.table({
        'user_url': vc.field('values'),
        'lang': pa.array([lang] * n_users_lang, type=pa.string()),
        'count': vc.field('counts'),
    })
    long_tables.append(lang_tbl)
    del vc

    print(f'{lang:>8}: {n_users_lang:>10,} English users also wrote in {lang!r}')

long_tbl = pa.concat_tables(long_tables)
del long_tables
print(f'\nTotal (user, language) pairs with count > 0: {long_tbl.num_rows:,}')

## 4. Consolidation: one record per user

Builds the `lang_counts` column as `map<string, int64>`, the native Parquet/Arrow map type, instead of a wide one-column-per-language layout.

In [ ]:
long_df = long_tbl.to_pandas()
del long_tbl

long_df = long_df.sort_values('user_url', kind='mergesort', ignore_index=True)

grouped = long_df.groupby('user_url', sort=False)
group_sizes = grouped.size()
total_reviews = grouped['count'].sum()

assert (group_sizes.index == total_reviews.index).all(), 'grupos desalinhados'
assert len(group_sizes) == len(en_users), 'not every English user appeared in the long format'
assert group_sizes.values.sum() == len(long_df), 'offsets do not cover all rows'

offsets = np.zeros(len(group_sizes) + 1, dtype='int32')
np.cumsum(group_sizes.values, out=offsets[1:])

keys_array = pa.array(long_df['lang'].values, type=pa.string())
items_array = pa.array(long_df['count'].values.astype('int64'))
map_array = pa.MapArray.from_arrays(offsets, keys_array, items_array)

result_table = pa.table({
    'user_url': pa.array(group_sizes.index.values, type=pa.string()),
    'lang_counts': map_array,
    'total_reviews': pa.array(total_reviews.values.astype('int64')),
})

print(result_table.schema)
print(f'Users in the final result: {result_table.num_rows:,}')

## 5. Verification

Two independent checks of the counts above (they do not reuse the intermediate objects already computed):

1. For a sample of multilingual users, recompute the per-language counts from scratch.
2. Check that the map values sum to `total_reviews` for every user.

In [ ]:
# Check 1: recompute from scratch the counts of a few multilingual users
result_df = result_table.to_pandas()
n_langs_per_user = result_df['lang_counts'].apply(len)
print('Distribution of the number of languages per English user:')
print(n_langs_per_user.value_counts().sort_index())

sample = (
    result_df[n_langs_per_user >= 3]
    .sample(n=min(5, (n_langs_per_user >= 3).sum()), random_state=42)
)

for _, row in sample.iterrows():
    user = row['user_url']
    counts_saved = dict(row['lang_counts'])
    for lang, expected_count in counts_saved.items():
        lang_dir = REVIEWS_DATA / f'review_lang={lang}'
        files = sorted(lang_dir.glob('*.parquet'))
        n_matches = 0
        for f in files:
            tbl = pq.read_table(f, columns=['user_url'])
            n_matches += pc.sum(pc.equal(tbl.column('user_url'), user)).as_py() or 0
        assert n_matches == expected_count, (
            f'{user} / {lang}: saved={expected_count} recomputed={n_matches}'
        )
    print(f'OK {user}: {counts_saved}')

print('\nChecagem 1 (recomputo independente): OK')

In [ ]:
# Check 2: the map values sum to total_reviews, for all users
map_sum = result_df['lang_counts'].apply(lambda d: sum(v for _, v in d))
assert (map_sum == result_df['total_reviews']).all(), 'map sum differs from total_reviews'

# Every English user must have the 'en' key in the map, with count >= 1
has_en = result_df['lang_counts'].apply(lambda d: dict(d).get('en', 0) >= 1)
assert has_en.all(), 'there is an English user with no count in en'

print('Check 2 (map sum == total_reviews, and "en" key present in all): OK')

## 6. Descriptive statistics

In [ ]:
print(f'English users                       : {len(result_df):,}')
print(f'  wrote only in English               : {(n_langs_per_user == 1).sum():,} ({(n_langs_per_user == 1).mean() * 100:.2f}%)')
print(f'  wrote in >= 2 languages             : {(n_langs_per_user >= 2).sum():,} ({(n_langs_per_user >= 2).mean() * 100:.2f}%)')
print()
print('total_reviews (per user) - describe:')
print(result_df['total_reviews'].describe())
print()
print('Final result sample:')
result_df.head(10)

## 7. Saving the result

In [ ]:
EN_USERS_LANG_COUNTS_FILE.parent.mkdir(parents=True, exist_ok=True)
pq.write_table(result_table, EN_USERS_LANG_COUNTS_FILE)
print(f'Saved to: {EN_USERS_LANG_COUNTS_FILE}')

# Confirm that reading it back preserves the map
check = pd.read_parquet(EN_USERS_LANG_COUNTS_FILE).head(3)
check

## 8. CDFs

Two distributions over the population of users with at least one English review (`result_df`, computed in sections 4 and 5):

1. Number of distinct languages used per user.
2. Percentage of each user's reviews written in English.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

# Register the system's Times New Roman fonts (macOS does not ship them in the matplotlib cache)
for _dir in ['/Library/Fonts', '/System/Library/Fonts', os.path.expanduser('~/Library/Fonts')]:
    if os.path.isdir(_dir):
        for _fname in os.listdir(_dir):
            if 'Times' in _fname and _fname.lower().endswith(('.ttf', '.otf', '.ttc')):
                fm.fontManager.addfont(os.path.join(_dir, _fname))

FONTSIZE = 25
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': FONTSIZE,
})
FONT_PROP = fm.FontProperties(family='Times New Roman', size=FONTSIZE)

def _apply_font(ax):
    ax.set_xlabel(ax.get_xlabel(), fontproperties=FONT_PROP)
    ax.set_ylabel(ax.get_ylabel(), fontproperties=FONT_PROP)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(FONT_PROP)

### 8.1 CDF: number of languages used per user

In [ ]:
n_langs = result_df['lang_counts'].apply(len)

x = np.sort(n_langs.values)
y = np.arange(1, len(x) + 1) / len(x)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x, y, linewidth=2)
ax.set_xlabel('Number of languages used per user')
ax.set_ylabel('CDF')
ax.set_xticks(range(1, int(n_langs.max()) + 1))
ax.grid(True, linewidth=1, color='lightgray')
ax.set_facecolor('white')
_apply_font(ax)
plt.tight_layout()
plt.show()

print(f'Median : {np.median(x):.0f} language(s)')
print(f'Mean   : {x.mean():.2f} languages')
print(f'Max    : {int(x.max())} languages')

### 8.2 CDF: percentage of reviews written in English, per user

In [ ]:
en_counts = result_df['lang_counts'].apply(lambda d: dict(d).get('en', 0))
pct_en = en_counts / result_df['total_reviews'] * 100

x2 = np.sort(pct_en.values)
y2 = np.arange(1, len(x2) + 1) / len(x2)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x2, y2, linewidth=2)
ax.set_xlabel('% of reviews written in English (per user)')
ax.set_ylabel('CDF')
ax.set_xlim(0, 100)
ax.grid(True, linewidth=1, color='lightgray')
ax.set_facecolor('white')
_apply_font(ax)
plt.tight_layout()
plt.show()

print(f'Median                           : {np.median(x2):.2f}%')
print(f'Mean                             : {x2.mean():.2f}%')
print(f'% of users 100% English          : {(pct_en == 100).mean() * 100:.2f}%')
print(f'% of users with <50% English     : {(pct_en < 50).mean() * 100:.2f}%')